# WISE and 21-cm HI Analysis of NGC 7331

This project combines infrared and radio observations to study galaxy luminosity, dust, star formation, neutral gas distribution, and kinematics.

## Scientific motivation

WISE infrared measurements sample stellar and warm-dust emission. THINGS 21-cm observations trace neutral hydrogen in NGC 7331. Together, they provide complementary views of a galaxy across the electromagnetic spectrum.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS

BASE = Path("..")
DATA = BASE / "data"
FITS_DIR = BASE / "FITS"
RESULTS = BASE / "results"
RESULTS.mkdir(exist_ok=True)

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.25})

## 1. WISE infrared catalogue

WISE observations at W1, W2, W3, and W4 sample 3.4, 4.6, 12, and 22 micrometre emission. These bands trace old stellar populations, warm dust, PAHs, and star-forming or AGN-heated dust.

In [ ]:
wise = pd.read_csv(DATA / "WISE_set1.csv")
print(f"Catalogue sources: {len(wise):,}")
print("Available columns:", ", ".join(wise.columns))
wise.head()

## 2. Converting magnitude to luminosity

Absolute magnitude is converted using the Pogson relation:

$$L = L_{\odot}\,10^{(M_{\odot}-M)/2.5}$$

In [ ]:
M_sun = 4.83
L_sun = 3.83e26

for band in ["W1", "W2", "W3", "W4"]:
    wise[f"Luminosity_{band}"] = L_sun * 10 ** ((M_sun - wise[f"{band}mag_x"]) / 2.5)

wise[["Source_id", "Luminosity_W1", "Luminosity_W2", "Luminosity_W3", "Luminosity_W4"]].head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, band in zip(axes.flat, ["W1", "W2", "W3", "W4"]):
    ax.scatter(wise[f"{band}mag_x"], wise[f"Luminosity_{band}"], s=8, alpha=0.45, color="steelblue")
    ax.set(xlabel=f"Absolute magnitude ({band})", ylabel="Luminosity [W]", yscale="log")
    ax.set_title(f"{band}: magnitude versus luminosity")
fig.suptitle("WISE magnitude-luminosity relationships", fontsize=15)
fig.tight_layout()
plt.show()

The plots show the expected inverse exponential relationship: more negative absolute magnitudes correspond to substantially higher luminosities.

## 3. Apparent magnitude and luminosity distance

The distance modulus predicts apparent magnitude from absolute magnitude and luminosity distance:

$$m = M + 5\log_{10}(d/10\;\mathrm{pc})$$

The catalogue provides no independent observed apparent-magnitude column, so this project calculates expected values only.

In [ ]:
pc_to_m = 3.086e16
wise["Distance_pc"] = wise["D_L"] / pc_to_m
for band in ["W1", "W2", "W3", "W4"]:
    wise[f"AppMag_{band}_expected"] = wise[f"{band}mag_x"] + 5 * np.log10(wise["Distance_pc"] / 10)
wise[["Source_id", "Distance_pc", "AppMag_W1_expected"]].head()

## 4. Flux and distance

For an isotropic source, observed flux follows the inverse-square law:

$$F = L/(4\pi d^2)$$

In [ ]:
for band in ["W1", "W2", "W3", "W4"]:
    wise[f"Flux_{band}"] = wise[f"Luminosity_{band}"] / (4 * np.pi * wise["D_L"]**2)

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, band in zip(axes.flat, ["W1", "W2", "W3", "W4"]):
    ax.scatter(wise["Distance_pc"], wise[f"Flux_{band}"], s=8, alpha=0.35, color="coral")
    ax.set(xscale="log", yscale="log", xlabel="Distance [pc]", ylabel=f"Flux ({band}) [W/m^2]")
    ax.set_title(f"{band}: flux versus distance")
fig.suptitle("WISE flux trends across luminosity distance", fontsize=15)
fig.tight_layout()
plt.show()

The complete population has substantial scatter because sources have different intrinsic luminosities. Restricting the sample to a narrow absolute-magnitude range isolates the inverse-square trend more clearly.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, band in zip(axes.flat, ["W1", "W2", "W3", "W4"]):
    median_magnitude = wise[f"{band}mag_x"].median()
    subset = wise[np.abs(wise[f"{band}mag_x"] - median_magnitude) <= 1]
    ax.scatter(subset["Distance_pc"], subset[f"Flux_{band}"], s=12, alpha=0.65, color="seagreen", label="Comparable luminosity")
    d = np.logspace(np.log10(subset["Distance_pc"].min()), np.log10(subset["Distance_pc"].max()), 200)
    f = d**-2
    f *= subset[f"Flux_{band}"].median() / np.median(f)
    ax.plot(d, f, "k--", linewidth=1.5, label=r"$F \propto d^{-2}$")
    ax.set(xscale="log", yscale="log", xlabel="Distance [pc]", ylabel=f"Flux ({band}) [W/m^2]")
    ax.set_title(f"{band}: controlled luminosity sample")
    ax.legend(fontsize=8)
fig.suptitle("Flux-distance behaviour for sources with similar absolute magnitude", fontsize=15)
fig.tight_layout()
plt.show()

## 5. Infrared colour-colour diagnostics

W1-W2 is sensitive to hot dust and AGN emission, while W2-W3 traces PAH and warm-dust content associated with star formation.

In [ ]:
wise["C1_W1_W2"] = wise["W1mag_x"] - wise["W2mag_x"]
wise["C2_W2_W3"] = wise["W2mag_x"] - wise["W3mag_x"]
fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(wise["C2_W2_W3"], wise["C1_W1_W2"], s=10, alpha=0.45, color="steelblue")
ax.axhline(0.5, color="crimson", linestyle="--", linewidth=1.5, label="AGN criterion: W1-W2 > 0.5")
ax.set(xlabel="W2-W3 colour", ylabel="W1-W2 colour", title="WISE colour-colour diagram")
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS / "wise_colour_colour.png", dpi=200, bbox_inches="tight")
plt.show()

High W1-W2 sources are candidates for hot-dust or AGN-dominated emission. Larger W2-W3 colours tend to indicate stronger PAH and warm-dust contributions. Colour thresholds are diagnostic rather than definitive classifications.

## 6. HI moment maps of NGC 7331

Moment maps summarize a 21-cm spectral cube:

- **Moment 0:** integrated neutral-hydrogen intensity
- **Moment 1:** intensity-weighted line-of-sight velocity
- **Moment 2:** intensity-weighted velocity dispersion

In [ ]:
def fit_shape_and_extract(filename):
    hdul = fits.open(FITS_DIR / filename)
    data = hdul[0].data[0, 0, :, :]
    header = hdul[0].header
    hdul.close()
    return data, header

In [ ]:
mom0, header0 = fit_shape_and_extract("NGC_7331_NA_MOM0_THINGS.FITS")
fig = plt.figure(figsize=(7, 7))
ax = plt.subplot(projection=WCS(header0, naxis=2))
im = ax.imshow(mom0, origin="lower", cmap="Spectral_r", vmax=np.nanpercentile(mom0, 99))
ax.set(xlabel="Right Ascension (J2000)", ylabel="Declination (J2000)", title="NGC 7331 - Neutral Hydrogen Distribution")
cb = fig.colorbar(im, ax=ax, shrink=0.82)
cb.set_label("Integrated intensity")
fig.tight_layout()
fig.savefig(RESULTS / "ngc7331_moment0.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
mom1, header1 = fit_shape_and_extract("NGC_7331_NA_MOM1_THINGS.FITS")
velocity_kms = mom1 / 1000.0
fig = plt.figure(figsize=(7, 7))
ax = plt.subplot(projection=WCS(header1, naxis=2))
im = ax.imshow(velocity_kms, origin="lower", cmap="Spectral_r", vmin=np.nanpercentile(velocity_kms, 1), vmax=np.nanpercentile(velocity_kms, 99))
levels = np.arange(np.floor(np.nanmin(velocity_kms) / 100) * 100, np.ceil(np.nanmax(velocity_kms) / 100) * 100 + 100, 100)
ax.contour(velocity_kms, levels=levels, colors="black", linewidths=0.45, alpha=0.7)
ax.set(xlabel="Right Ascension (J2000)", ylabel="Declination (J2000)", title="NGC 7331 - HI Velocity Field")
cb = fig.colorbar(im, ax=ax, shrink=0.82)
cb.set_label("Line-of-sight velocity [km/s]")
fig.tight_layout()
fig.savefig(RESULTS / "ngc7331_moment1.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
mom2, header2 = fit_shape_and_extract("NGC_7331_NA_MOM2_THINGS.FITS")
dispersion_kms = mom2 / 1000.0
fig = plt.figure(figsize=(7, 7))
ax = plt.subplot(projection=WCS(header2, naxis=2))
im = ax.imshow(dispersion_kms, origin="lower", cmap="Purples_r", vmax=np.nanpercentile(dispersion_kms, 99))
ax.set(xlabel="Right Ascension (J2000)", ylabel="Declination (J2000)", title="NGC 7331 - HI Velocity Dispersion")
cb = fig.colorbar(im, ax=ax, shrink=0.82)
cb.set_label("Velocity dispersion [km/s]")
fig.tight_layout()
fig.savefig(RESULTS / "ngc7331_moment2.png", dpi=200, bbox_inches="tight")
plt.show()

## Results and interpretation

- WISE measurements confirm the logarithmic magnitude-luminosity relationship.
- Comparing sources with similar absolute magnitudes exposes the inverse-square flux trend.
- Infrared colours identify populations dominated by stellar, dusty, star-forming, or AGN-heated emission.
- The HI moment 0 map reveals the galaxy gas distribution.
- The moment 1 map reveals ordered rotation across the disc.
- The moment 2 map identifies regions of enhanced gas-motion broadening.

Together, these analyses demonstrate how multiwavelength observations characterize a galaxy's energy sources, dust, gas structure, and kinematics.